In [2]:
## =========================================================
# 0. IMPORT
# =========================================================
import pandas as pd
import numpy as np
import mindspore as ms
from mindspore import Tensor, context, load_checkpoint, load_param_into_net
from mindspore import nn
import joblib
import moxing as mox
import os
import time

context.set_context(mode=context.GRAPH_MODE, device_target="CPU")

# =========================================================
# 1. PATH
# =========================================================
PATH_MODEL_OBS   = 'obs://mindspore-dataset-1/output_model2/model_lstm.ckpt'
PATH_SCALER_OBS  = 'obs://mindspore-dataset-1/output_model2/scaler.save'

LOCAL_CKPT       = './model_lstm.ckpt'
LOCAL_SCALER     = './scaler.save'

PATH_SENSOR_OBS  = 'obs://mindspore-dataset-1/Data/Sensor_Terkini.csv'
PATH_OUTPUT_OBS  = 'obs://mindspore-dataset-1/Prediction/hasil_prediksi_simulasi.csv'

# =========================================================
# 2. DOWNLOAD MODEL & SCALER
# =========================================================
if not os.path.exists(LOCAL_CKPT):
    print("📥 Download model...")
    mox.file.copy(PATH_MODEL_OBS, LOCAL_CKPT)

if not os.path.exists(LOCAL_SCALER):
    print("📥 Download scaler...")
    mox.file.copy(PATH_SCALER_OBS, LOCAL_SCALER)

print("📂 File lokal:", os.listdir('.'))

# =========================================================
# 3. MODEL (DIOTIMALKAN BERDASARKAN lstm.py)
# =========================================================
class LSTMModel(nn.Cell):
    def __init__(self, input_size, hidden_size, dense_units, output_size, keep_prob=0.9):
        super().__init__()

        # Layer LSTM 1 & 2 ditumpuk vertikal dengan num_layers=2
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True)
        self.dropout = nn.Dropout(keep_prob=keep_prob)
        
        # Hidden Layer Dense 1
        self.fc1 = nn.Dense(hidden_size, dense_units)
        self.relu1 = nn.ReLU()
        
        # Hidden Layer Dense 2 (Dimensi berkurang setengah)
        self.fc2 = nn.Dense(dense_units, dense_units // 2)
        self.relu2 = nn.ReLU()
        
        # Output Layer (Menerima input dari fc2 menuju target 90 unit flat)
        self.fc3 = nn.Dense(dense_units // 2, output_size)

    def construct(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]  # Ambil timestep terakhir

        # Proses lewat Hidden Layer Dense 1 + Dropout
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu1(x)
        
        # Proses lewat Hidden Layer Dense 2
        x = self.fc2(x)
        x = self.relu2(x)
        
        # Proses ke Output Layer
        x = self.fc3(x)

        return x

# --- SETTING HYPERPARAMETER (IDENTIK DENGAN lstm.py) ---
INPUT_SIZE   = 7    # 5 kolom sensor + 2 kolom waktu (sin/cos)
HIDDEN_SIZE  = 128      
DENSE_UNITS  = 256     
DROPOUT_PROB = 0.9    
OUTPUT_SIZE  = 90   # 18 langkah waktu x 5 fitur sensor = 90 outputs
# -------------------------------------------------------

# Inisialisasi model dengan cetakan parameter terstandar
model = LSTMModel(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    dense_units=DENSE_UNITS,
    output_size=OUTPUT_SIZE,
    keep_prob=DROPOUT_PROB
)

# =========================================================
# 4. LOAD MODEL
# =========================================================
print("🧠 Load checkpoint...")
param_dict = load_checkpoint(LOCAL_CKPT)
load_param_into_net(model, param_dict)

model.set_train(False) # Kunci model ke mode evaluasi murni (menonaktifkan dropout internal)
print("✅ Model siap!")

# =========================================================
# 5. LOAD SCALER
# =========================================================
scaler = joblib.load(LOCAL_SCALER)
print("✅ Scaler siap!")

# =========================================================
# 6. BUFFER CONFIGURATION
# =========================================================
data_buffer = []
fitur_sensor = ['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']
fitur_input = fitur_sensor + ['hour_sin', 'hour_cos']

print("🧠 Mesin aktif...")

# =========================================================
# 7. LOOP REAL-TIME
# =========================================================
while True:
    try:
        # =========================
        # Ambil data terbaru
        # =========================
        mox.file.copy(PATH_SENSOR_OBS, './temp.csv')
        df_now = pd.read_csv('./temp.csv')

        new_entry = df_now.iloc[-1].to_dict()

        if not data_buffer or new_entry['Jam'] != data_buffer[-1]['Jam']:
            data_buffer.append(new_entry)
            print(f"📥 {new_entry['Jam']} | Suhu {new_entry['Suhu']:.2f}")

        # =========================
        # Jika buffer cukup (SEQ_LENGTH = 18)
        # =========================
        if len(data_buffer) >= 18:

            if len(data_buffer) > 18:
                data_buffer.pop(0)

            df_predict = pd.DataFrame(data_buffer)

            # =========================
            # Feature engineering
            # =========================
            df_predict['Datetime'] = pd.to_datetime(
                df_predict['Tanggal'] + ' ' + df_predict['Jam'],
                dayfirst=True
            )

            df_predict['hour_sin'] = np.sin(2 * np.pi * df_predict['Datetime'].dt.hour / 24)
            df_predict['hour_cos'] = np.cos(2 * np.pi * df_predict['Datetime'].dt.hour / 24)

            # Smoothing tanpa merusak atau mereduksi dimensi baris
            df_predict[fitur_sensor] = df_predict[fitur_sensor].rolling(
                window=3,
                min_periods=1
            ).mean()

            # =========================
            # Scaling Input (7 Dimensi)
            # =========================
            data_scaled = scaler.transform(df_predict[fitur_input].values)
            input_tensor = Tensor(np.array([data_scaled]), ms.float32)

            # =========================
            # Prediksi dengan Jaringan Grafik MindSpore
            # =========================
            pred = model(input_tensor).asnumpy()
            pred = pred.reshape(18, 5) # Kembalikan bentuk flat 90 ke bentuk dimensi matriks (18, 5)

            # =========================
            # Inverse scaling
            # =========================
            dummy = np.zeros((18, len(fitur_input)))
            dummy[:, :5] = pred

            hasil = scaler.inverse_transform(dummy)[:, :5]
            hasil = np.maximum(0, hasil) # Mengunci batas fisik agar sensor tidak bernilai negatif

            # =========================
            # Generate waktu masa depan (Proyeksi 3 Jam ke depan tiap kelipatan 10 menit)
            # =========================
            waktu_akhir = df_predict['Datetime'].iloc[-1]

            list_waktu = [
                waktu_akhir + pd.Timedelta(minutes=10*(i+1))
                for i in range(18)
            ]

            df_hasil = pd.DataFrame(
                hasil,
                columns=fitur_sensor
            )

            df_hasil['Tanggal'] = [t.strftime('%d/%m/%Y') for t in list_waktu]
            df_hasil['Jam'] = [t.strftime('%H:%M:%S') for t in list_waktu]

            df_hasil = df_hasil[
                ['Tanggal', 'Jam'] + fitur_sensor
            ]

            # =========================
            # Save lokal & upload balik ke Huawei Cloud OBS
            # =========================
            df_hasil.to_csv('./hasil.csv', index=False)
            mox.file.copy('./hasil.csv', PATH_OUTPUT_OBS)

            print(f"🔮 Prediksi Sukses! Rekomendasi kondisi 3 jam kedepan diunggah ke OBS.")

        else:
            print(f"⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: {len(data_buffer)}/18")

        time.sleep(1)

    except Exception as e:
        print("⚠️ Perhatian Sistem, Terjadi Eror:", e)
        time.sleep(5)

INFO:root:Using MoXing-v2.1.0.5d9c87c8-5d9c87c8
INFO:root:Using OBS-Python-SDK-3.20.9.1


📂 File lokal: ['simulaktuator.ipynb', 'lost+found', 'data_bersih.csv', 'sensor_terkini.csv', '.lsp_symlink', '__pycache__', 'lstm_transformer.ckpt', 'training_transformers.ipynb', 'model_lstm.ckpt', '.ipynb_checkpoints', 'hasil.csv', '.virtual_documents', 'lstm.py', 'transformers.py', '.Trash-1000', '.modelarts', 'temp.csv', 'transformer_results.csv', 'scaler.save', 'model.ckpt', 'SimulLstm.ipynb', 'notebook_training.ipynb']
🧠 Load checkpoint...
✅ Model siap!
✅ Scaler siap!
🧠 Mesin aktif...


/home/ma-user/anaconda3/envs/MindSpore/lib/python3.7/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.12) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


📥 23:35:02 | Suhu 27.30
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 1/18
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 1/18
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 1/18
📥 23:36:14 | Suhu 30.10
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 2/18
📥 23:36:16 | Suhu 29.75
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 3/18
📥 23:36:17 | Suhu 32.00
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 4/18
📥 23:36:18 | Suhu 32.10
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 5/18
📥 23:36:19 | Suhu 32.10
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 6/18
📥 23:36:20 | Suhu 32.20
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 7/18
📥 23:36:21 | Suhu 32.20
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 8/18
📥 23:36:22 | Suhu 25.90
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 9/18
📥 23:36:24 | Suhu 29.45
⏳ Mengumpulkan data ke dalam jendela sekuens... Buffer: 10/18
📥 23:36:25 | Suhu 29.30
⏳ M

KeyboardInterrupt: 